# Ноутбук 02: Классификация текста — IMDB Sentiment Analysis

**Задача:** определить тональность рецензий на фильмы (позитивная / негативная).

**Датасет:** [IMDB Large Movie Review Dataset](https://huggingface.co/datasets/imdb)
- 25 000 рецензий для обучения, 25 000 для теста
- Метки: `0` (негативная) и `1` (позитивная)
- Реальные отзывы с IMDb.com

**Модель:** `distilbert-base-uncased-finetuned-sst-2-english`
- DistilBERT = сжатый BERT (в 2× меньше, в 2× быстрее, 97% качества BERT)
- Дообучен на SST-2 (Stanford Sentiment Treebank)

**Что будем делать:**
1. Загрузить датасет IMDB
2. Изучить данные
3. Запустить inference на тестовой выборке
4. Посчитать метрики качества
5. Посмотреть на ошибки модели

In [ ]:
# Импорты
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from collections import Counter

print('Версия torch:', torch.__version__)
print('GPU доступен:', torch.cuda.is_available())

## Шаг 1: Загрузка датасета IMDB

In [ ]:
# Загружаем датасет IMDB через HuggingFace datasets
# При первом запуске скачает ~80 MB
dataset = load_dataset('imdb')

print('Структура датасета:')
print(dataset)
print()
print(f'Обучающих примеров: {len(dataset["train"])}')
print(f'Тестовых примеров:  {len(dataset["test"])}')

In [ ]:
# Смотрим на первые примеры
print('=== Позитивная рецензия (label=1) ===')
pos_example = dataset['train'][0]
print(f'Метка: {pos_example["label"]} (1 = позитивная)')
print(f'Текст: {pos_example["text"][:300]}...')
print()

# Найдём негативную
neg_example = next(x for x in dataset['train'] if x['label'] == 0)
print('=== Негативная рецензия (label=0) ===')
print(f'Метка: {neg_example["label"]} (0 = негативная)')
print(f'Текст: {neg_example["text"][:300]}...')

In [ ]:
# Статистика датасета
train_labels = dataset['train']['label']
label_counts = Counter(train_labels)

print('Распределение классов (train):')
print(f'  Позитивных (1): {label_counts[1]:,} ({label_counts[1]/len(train_labels):.1%})')
print(f'  Негативных (0): {label_counts[0]:,} ({label_counts[0]/len(train_labels):.1%})')
print()

# Длина рецензий
lengths = [len(text.split()) for text in dataset['train']['text'][:1000]]
print(f'Длина рецензий (первые 1000):')
print(f'  Средняя: {np.mean(lengths):.0f} слов')
print(f'  Медиана: {np.median(lengths):.0f} слов')
print(f'  Максимум: {max(lengths)} слов')

## Шаг 2: Понимаем токенизацию

In [ ]:
# Загружаем токенизатор отдельно — чтобы увидеть, что происходит
model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Токенизируем простой пример
text = 'This movie was absolutely amazing! I loved every minute.'

# Что делает токенизатор:
tokens = tokenizer.tokenize(text)         # разбивает на токены
encoded = tokenizer.encode(text)          # токены → числа (IDs)
decoded = tokenizer.decode(encoded)       # числа → текст обратно

print(f'Исходный текст: "{text}"')
print()
print(f'Токены ({len(tokens)} шт.): {tokens}')
print()
print(f'IDs:            {encoded}')
print()
print(f'Назад:          "{decoded}"')
print()
print('Обратите внимание:')
print('  [CLS] (101) — специальный токен начала')
print('  [SEP] (102) — специальный токен конца')
print('  "amazing" → ["amazing"] — одно слово = один токен')
print('  "##" — продолжение предыдущего токена (субслово)')

## Шаг 3: Inference на тестовой выборке

In [ ]:
# Создаём pipeline для классификации
classifier = pipeline(
    'sentiment-analysis',
    model=model_name,
    device=0 if torch.cuda.is_available() else -1  # GPU если есть, иначе CPU
)

# Берём 200 примеров из тестового датасета для быстрой оценки
# (полный тест из 25K займёт несколько минут на CPU)
test_sample = dataset['test'].select(range(200))

print(f'Оцениваем модель на {len(test_sample)} тестовых примерах...')

In [ ]:
# Запускаем inference батчами
# batch_size=16 — обрабатываем 16 текстов за раз (быстрее, чем по одному)

texts = test_sample['text']
true_labels = test_sample['label']  # 0 или 1

# Модель возвращает POSITIVE/NEGATIVE, переводим в числа
predictions_raw = classifier(texts, batch_size=16, truncation=True, max_length=512)

# Конвертируем: 'POSITIVE' → 1, 'NEGATIVE' → 0
pred_labels = [1 if r['label'] == 'POSITIVE' else 0 for r in predictions_raw]
pred_scores = [r['score'] for r in predictions_raw]

print('Inference завершён!')
print(f'Первые 5 предсказаний: {pred_labels[:5]}')
print(f'Первые 5 реальных:     {list(true_labels[:5])}')

## Шаг 4: Метрики качества

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

# Базовые метрики
accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)

print(f'Accuracy: {accuracy:.1%}')
print(f'F1 Score: {f1:.3f}')
print()

# Детальный отчёт по классам
print('Детальный отчёт:')
print(classification_report(
    true_labels, pred_labels,
    target_names=['NEGATIVE', 'POSITIVE']
))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(true_labels, pred_labels)

print('Матрица ошибок:')
print(f'             Предсказано:')
print(f'             NEGATIVE  POSITIVE')
print(f'Реально NEG: {cm[0,0]:>6}    {cm[0,1]:>6}   (TP для NEG, FP для POS)')
print(f'Реально POS: {cm[1,0]:>6}    {cm[1,1]:>6}   (FN для NEG, TP для POS)')
print()
print(f'Верно классифицировано: {cm[0,0] + cm[1,1]} из {len(true_labels)}')
print(f'Ошибочно:               {cm[0,1] + cm[1,0]} из {len(true_labels)}')

## Шаг 5: Анализ ошибок — где модель ошибается?

In [ ]:
# Найдём примеры, где модель ошиблась
errors = [
    {
        'text': texts[i],
        'true': true_labels[i],
        'pred': pred_labels[i],
        'score': pred_scores[i]
    }
    for i in range(len(texts))
    if pred_labels[i] != true_labels[i]
]

print(f'Всего ошибок: {len(errors)}')
print()

# Показываем первые 3 ошибки
label_names = {0: 'NEGATIVE', 1: 'POSITIVE'}
for i, err in enumerate(errors[:3], 1):
    print(f'--- Ошибка {i} ---')
    print(f'Реальная метка:  {label_names[err["true"]]}')
    print(f'Предсказание:    {label_names[err["pred"]]} (уверенность: {err["score"]:.1%})')
    print(f'Текст: {err["text"][:200]}...')
    print()

In [ ]:
# Тест на своих текстах — попробуйте!
my_texts = [
    'The acting was superb but the plot was quite boring and predictable.',  # неоднозначный
    'Worst film ever made. Complete waste of time.',
    'A masterpiece of cinema. Highly recommended!'
]

results = classifier(my_texts)
for text, result in zip(my_texts, results):
    print(f'{result["label"]:10s} ({result["score"]:.1%}): "{text}"')